# Reproducible Research with Verdifax

This notebook shows how to take a small machine-learning analysis from
*"it runs on my laptop"* to *"cryptographically attested, byte-identical
on replay, bound to a declared environment fingerprint."*

The Verdifax pipeline does three things you can't easily do yourself:

1. **Canonical seal** — your result is normalized into a deterministic
   form (RFC 8785 JCS) and hashed. Anyone who has the same input data,
   the same code, and the same declared environment can independently
   recompute that hash and check it matches.
2. **Reproducibility-context binding** — the orchestrator binds your
   *declared* runtime fingerprint (Python version, pinned dependencies,
   container image hash, git SHA, random seeds, platform) into the
   audit bundle as Category 6. The seal is over your declaration, not
   a guess.
3. **Transparency log anchor** — when running against a Rekor-anchored
   deployment, every attestation is published to the Sigstore
   transparency log, so erasing or rewriting the record after the fact
   is detectable.

## What you'll need

- A Verdifax API key (`VERDIFAX_API_KEY` environment variable).
- `pip install verdifax scikit-learn`.

## 1. Set up a deterministic analysis

We'll fit a logistic regression on a tiny synthetic dataset, with a
fixed random seed. This is intentionally trivial — the point is the
attestation, not the model.

In [ ]:
import json
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

RNG_SEED = 42

X, y = make_classification(
    n_samples=200,
    n_features=4,
    n_informative=3,
    random_state=RNG_SEED,
)
model = LogisticRegression(random_state=RNG_SEED).fit(X, y)

# Capture the result in a canonical, deterministic JSON form.
result = {
    "coefficients": [round(c, 8) for c in model.coef_[0].tolist()],
    "intercept": round(float(model.intercept_[0]), 8),
    "training_accuracy": round(float(model.score(X, y)), 8),
    "n_samples": int(X.shape[0]),
    "n_features": int(X.shape[1]),
}
payload = json.dumps(result, sort_keys=True, separators=(",", ":"))
print(payload)

## 2. Capture the research environment

`capture_environment()` auto-detects the Python runtime version,
every package version installed in the current interpreter
(via `importlib.metadata`), the git commit SHA of the working
directory (best-effort), the container image hash (from
`/proc/self/cgroup` on Linux, best-effort), and the platform
descriptor. You declare any random seeds explicitly so the
orchestrator records exactly which seeds were claimed.

Auto-detection failures silently leave the field `None`. The
orchestrator records `None` as *"not declared"* rather than
fabricating a claim.

In [ ]:
from verdifax.research import capture_environment

ctx = capture_environment(declared_seeds={"numpy": RNG_SEED, "sklearn": RNG_SEED})
print("runtime:        ", ctx.runtime_name, ctx.runtime_version)
print("platform:       ", ctx.platform)
print("git_commit_sha: ", ctx.git_commit_sha)
print("# pinned deps:  ", len(ctx.pinned_dependencies or []))
print("declared seeds: ", ctx.random_seeds)
print("declared flag:  ", ctx.declared)

## 3. Attest the result

`VerdifaxClient.attest()` POSTs the payload to `/execute` and
binds the reproducibility context into the audit bundle as
**Category 6**. The returned manifest carries the canonical
ManifestHash — the cryptographic seal of this analysis.

In [ ]:
from verdifax import VerdifaxClient

client = VerdifaxClient()  # picks up VERDIFAX_API_URL / VERDIFAX_API_KEY from env

attestation = client.attest(
    payload=payload,
    program_id="0" * 64,                # registry-authorized program ID
    route_id="paper-figure-3",          # human-meaningful route label
    registry_record_hash="0" * 64,       # §0 record hash; placeholder in this demo
    reproducibility_context=ctx,
)
print("run_id:        ", attestation.run_id)
print("ManifestHash:  ", attestation.manifest.manifest_hash)

## 4. Prove the pipeline is deterministic

`verify_determinism()` runs the same payload through the pipeline
**twice** and reports whether both invocations produced byte-identical
canonical manifest hashes. The `deterministic` flag is grounded on
manifest-hash equality — the seal of the pipeline output. Bundle hash
differences (when surfaced) reflect server-observed timing variation
and are labeled as informational.

In [ ]:
from verdifax.research import verify_determinism

determinism = verify_determinism(
    client=client,
    payload=payload,
    program_id="0" * 64,
    route_id="paper-figure-3-verify",
    registry_record_hash="0" * 64,
    reproducibility_context=ctx,
)
print("deterministic:           ", determinism.deterministic)
print("first.manifest_hash:     ", determinism.first.manifest_hash)
print("second.manifest_hash:    ", determinism.second.manifest_hash)
print("manifest_hash_differs:   ", determinism.diff.manifest_hash_differs)
print("differing_fields (info): ", determinism.diff.differing_fields)
assert determinism.deterministic, "pipeline produced non-deterministic seals — this should never happen"

## 5. Re-run the notebook

If you restart the kernel and run all cells again *without changing the
input data, code, or declared environment*, the `ManifestHash` printed
in step 3 will be **byte-identical** to the previous run. That's the
core reproducibility guarantee Verdifax delivers: the same scientific
result always seals to the same cryptographic hash, and the seal is
bound to your declared environment fingerprint.

If anything *did* change — a dependency was bumped, a seed was
different, the input data was perturbed — the hash will differ, and
you (and any auditor, reviewer, or replicator) will know.

## What to do with the manifest hash

- **Cite it in the paper / report.** Treat it like a DOI for the
  computational result: short, unambiguous, machine-checkable.
- **Pin it in supplementary materials.** Any reviewer can
  independently recompute the manifest hash if they have the same
  inputs and declared environment.
- **Bind it to the artifact.** Sign your dataset or paper PDF over
  the manifest hash so the link is non-repudiable.

For audit-grade deployments, ask your administrator whether the
instance is **Rekor-anchored** — that adds an immutable Sigstore
transparency-log entry to every attestation, detectable to anyone
auditing the log.